# 3 · Train the detector - finds the dice in the image

Trains **ESPDet-Pico** (a very small detector by Espressif for the ESP32-S3) on your renders - the first time
**from scratch**; later (e.g. after more renders) it can **continue from an earlier run** (`START_FROM`), which is much
shorter and does not forget anything because the old images train along. It only learns “there is a die here” - which
symbol is on top is the job of the classifier (notebook 4). At the end the detector is prepared for the ESP32
(ONNX + image sets) and stored in `export/`.

**Before:** notebook 2 is done, *Runtime → Change runtime type → L4 GPU*.

**If Colab stops:** run everything again. Training continues from the last saved state
(the checkpoints are on Drive under `projects/…/detector/`).

## 1 · Settings
The trainer has filled in Drive folder and project. Normally nothing needs to be changed here.

In [ ]:
DRIVE_FOLDER = "__DRIVE__"        # folder in "My Drive"
PROJECT = "__PROJECT__"           # project folder name (under projects/)
NB_NAME, NB_VERSION = "3_detector", 2   # version of this notebook (to detect updates)
EPOCHS = 300                # from scratch: passes over all images (upper limit)
PATIENCE = 100              # from scratch: stop when nothing improves for this many epochs
BATCH = 64
RUN = "run1"                # name of the training run (a new name = a new run, the old one stays)
START_FROM = ""             # continue from the best.pt of this run; "" (or missing) = from scratch
EPOCHS_CONTINUE = 80        # when continuing: upper limit …
PATIENCE_CONTINUE = 25      # … and patience
DET_DATA = "/content/det_data"   # local (fast), rebuilt in every session

DRIVE = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
P = f"{DRIVE}/projects/{PROJECT}"
SCRIPTS = f"{DRIVE}/scripts"
SCENE = f"{P}/scene"
RENDERS = f"{P}/renders"
PREVIEW = f"{P}/renders_preview"
EXPORT = f"{P}/export"
DET = f"{P}/detector"
CLS = f"{P}/classifier"
ESP_DETECTION = "/content/esp-detection"
print("Project folder:", P)

## 2 · Check the GPU
Expected: **NVIDIA L4** (faster) or **Tesla T4**. If it says "no GPU": *Runtime → Change runtime type → L4 GPU*
(or T4), then start again from the top.

In [ ]:
import subprocess
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    gpu = ""
print("✔ GPU:", gpu) if gpu else print("✘ no GPU! Runtime → Change runtime type → L4 or T4 GPU, then run everything again.")

## 3 · Connect Google Drive and unpack the newest package
Colab asks whether the notebook may access your Drive → **Allow**.

Then the **newest Colab package** in `packages/` is unpacked (the ZIP file from the trainer - upload it to
*My Drive → __DRIVE__ → packages*, do **not** unpack it). Scripts, notebooks, `project.json`, photos, test images and the
camera setup are updated. **Renders and training results are not touched.**
A package is unpacked only once; a newer package replaces photos, test images and camera setup completely.

In [ ]:
import glob, json, os, shutil, zipfile
from google.colab import drive
drive.mount("/content/drive")
PACKAGES = f"{DRIVE}/packages"
os.makedirs(PACKAGES, exist_ok=True)
packages = sorted(glob.glob(f"{PACKAGES}/Colab_{PROJECT}_*.zip"))
state_file = f"{DRIVE}/package_state.json"
state = json.load(open(state_file)) if os.path.exists(state_file) else {}
if not packages:
    print(f"⚠ No package “Colab_{PROJECT}_….zip” in {PACKAGES} - create one in the trainer and upload the ZIP file there.")
elif state.get(PROJECT) == os.path.basename(packages[-1]):
    print("✔ newest package is unpacked already:", os.path.basename(packages[-1]))
else:
    new = packages[-1]
    print("unpacking:", os.path.basename(new))
    with zipfile.ZipFile(new) as z:
        names = [n for n in z.namelist() if not n.endswith("/")]
        prefix = names[0].split("/")[0]
        # mirror of the laptop: these folders are replaced completely (deleted photos disappear here too)
        for part in ("scripts", "notebooks", f"projects/{PROJECT}/photos", f"projects/{PROJECT}/test_images",
                     f"projects/{PROJECT}/camera_setup"):
            if any(n.startswith(f"{prefix}/{part}/") for n in names) and os.path.isdir(f"{DRIVE}/{part}"):
                shutil.rmtree(f"{DRIVE}/{part}")
        for n in names:
            target = os.path.join(DRIVE, n.split("/", 1)[1])
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(n) as src, open(target, "wb") as f:
                shutil.copyfileobj(src, f)
    state[PROJECT] = os.path.basename(new)
    json.dump(state, open(state_file, "w"), indent=1)
    print(f"✔ unpacked ({len(names)} files)")
    try:
        v = json.load(open(f"{DRIVE}/notebooks/version.json")).get(NB_NAME, 0)
        if v > NB_VERSION:
            print(f"⚠ The package contains a NEWER version of this notebook: please open notebooks/{NB_NAME}.ipynb in Drive again.")
    except (OSError, ValueError):
        pass
missing = [p for p in (f"{P}/project.json", f"{SCRIPTS}/render.py") if not os.path.exists(p)]
if missing:
    print("✘ not found:", *missing, sep="\n   ")
else:
    print("✔ project and scripts found")

## 4 · Install the training programs (about 1-2 minutes, every session)
Fixed versions so that the result stays reproducible: **ultralytics 8.4.189**, **albumentations 2.0.8**,
**onnx 1.23.2**, **onnxsim 0.8.1** and Espressif's **esp-detection** (fixed commit), which describes the detector
*ESPDet-Pico*. PyTorch comes with Colab.

In [ ]:
import os, subprocess, sys, json
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.189", "albumentations==2.0.8",
                "onnx==1.23.2", "onnxsim==0.8.1"], check=True)
if not os.path.isdir(ESP_DETECTION):
    subprocess.run(["git", "clone", "-q", "https://github.com/espressif/esp-detection.git", ESP_DETECTION], check=True)
subprocess.run(["git", "-C", ESP_DETECTION, "checkout", "-q", "28de5902a357d99dd9a5db0f52905255641c5a78"], check=True)
sys.path.insert(0, SCRIPTS)
import torch, ultralytics, ml_tools
d = ml_tools.load_project(P)
IMGSZ, CROP = ml_tools.sizes(d)
print(f"✔ torch {torch.__version__} (CUDA: {torch.cuda.is_available()}), ultralytics {ultralytics.__version__}")
print(f"✔ classes: {', '.join(ml_tools.classes(d))} · detector input {IMGSZ} px · crop {CROP} px")

## 5 · Build the dataset (about 10-15 minutes, in every new session)
Copies the renders (32 at a time, with progress) to the fast Colab disk and splits them into training (90 %) and
validation (10 %). Your real labelled throws from the trainer are added as **real test images** - they are never used
for training, only for measuring.

In [ ]:
import re, subprocess, time
t0 = time.time()
proc = subprocess.Popen(["python", f"{SCRIPTS}/det_dataset.py", "--project", P, "--renders", RENDERS, "--output", DET_DATA], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    z = re.sub(r"\x1b\[[0-9;]*[A-Za-z]", "", line.rstrip("\n").split("\r")[-1])   # no colour codes
    if z.strip() and (True):
        print(z, flush=True)
proc.wait()
print(f"⏱ {(time.time() - t0) / 60:.1f} min · return code {proc.returncode}")

## 6 · Timing: 1 epoch
Trains exactly one epoch and extrapolates how long `EPOCHS` would take (usually `PATIENCE` stops the training earlier).
Skipping this cell is fine. **Note:** the measurement includes the one-off start (filling the cache, building the
model) and therefore overestimates a lot. The real time per epoch is shown in cell 7.

In [ ]:
import shutil, time
START = f"{DET}/runs/{START_FROM}/weights/best.pt" if START_FROM else ""
CONTINUE = bool(START) and os.path.exists(START)
print(f"Run {RUN}: " + (f"continue from {START_FROM} (at most {EPOCHS_CONTINUE} epochs)" if CONTINUE
                        else f"from scratch (at most {EPOCHS} epochs)" + (f" - {START} is missing" if START else "")))
timing = f"{DET}/runs/timing"
if os.path.isdir(timing):
    shutil.rmtree(timing)
t0 = time.time()
r = subprocess.run(["python", f"{SCRIPTS}/det_train.py", "--data", f"{DET_DATA}/data.yaml", "--output", f"{DET}/runs",
                    "--name", "timing", "--imgsz", str(IMGSZ), "--epochs", "1", "--batch", str(BATCH),
                    "--esp-detection", ESP_DETECTION], capture_output=True, text=True)
s = time.time() - t0
print(r.stdout[-600:] if r.returncode == 0 else r.stdout[-2000:] + r.stderr[-2000:])
E = EPOCHS_CONTINUE if CONTINUE else EPOCHS
print(f"⏱ 1 epoch incl. start: {s:.0f} s  →  {E} epochs at most about {E * s / 3600:.1f} h "
      "(the start - filling the cache, building the model - happens only once, so the real time is lower)")

## 7 · Training (takes a while - see the timing)
After every epoch one line: the validation (`mAP50`, recall, precision - should go up, 1.0 would be perfect), the
duration of the epoch, the best epoch so far and how many epochs are left at most (`PATIENCE` stops earlier). Checkpoints are saved to Drive continuously.
**Stopped?** Just run all cells again - this cell resumes automatically.

In [ ]:
run = f"{DET}/runs/{RUN}"
if os.path.exists(f"{run}/result.json"):
    print(f"✔ Run “{RUN}” is finished already - for a new attempt change RUN above.")
else:
    START = f"{DET}/runs/{START_FROM}/weights/best.pt" if START_FROM else ""
    CONTINUE = bool(START) and os.path.exists(START)
    cmd = ["python", f"{SCRIPTS}/det_train.py", "--data", f"{DET_DATA}/data.yaml", "--output", f"{DET}/runs",
           "--name", RUN, "--imgsz", str(IMGSZ), "--batch", str(BATCH), "--esp-detection", ESP_DETECTION]
    cmd += (["--start-from", START, "--epochs", str(EPOCHS_CONTINUE), "--patience", str(PATIENCE_CONTINUE)] if CONTINUE
            else ["--epochs", str(EPOCHS), "--patience", str(PATIENCE)])
    if os.path.exists(f"{run}/weights/last.pt"):
        cmd.append("--resume")
        print("resuming from", f"{run}/weights/last.pt")
    import re, subprocess, time
    t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        z = re.sub(r"\x1b\[[0-9;]*[A-Za-z]", "", line.rstrip("\n").split("\r")[-1])   # no colour codes
        if z.strip() and (z.startswith("  epoch ") or "final validation" in z or "✔" in z or "⏱" in z or "rror" in z or "continuing" in z or "Resuming" in z):
            print(z, flush=True)
    proc.wait()
    print(f"⏱ {(time.time() - t0) / 60:.1f} min · return code {proc.returncode}")

## 8 · Measure the result (test only, no new training)
Measures the finished detector **every time anew** on the current test images - also when the training was skipped.
So you get the values on **real** images as soon as you have added them (trainer, new package).
**mAP50** (0-1): how well boxes and dice match. **Recall**: share of dice found. **Precision**: share of boxes that
really are dice. The values on **real** images matter most; a recall clearly below 0.95 there needs a closer look.
When continuing, `START_FROM` is measured on **the same** test images (before/after comparison).

In [ ]:
run = f"{DET}/runs/{RUN}"
r = subprocess.run(["python", f"{SCRIPTS}/det_train.py", "--data", f"{DET_DATA}/data.yaml", "--output", f"{DET}/runs",
                    "--name", RUN, "--imgsz", str(IMGSZ), "--esp-detection", ESP_DETECTION, "--test-only"],
                   capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-1500:] + r.stderr[-1500:])
res = json.load(open(f"{run}/result.json"))
before = None
if START_FROM and START_FROM != RUN and os.path.exists(f"{DET}/runs/{START_FROM}/weights/best.pt"):
    subprocess.run(["python", f"{SCRIPTS}/det_train.py", "--data", f"{DET_DATA}/data.yaml", "--output", f"{DET}/runs",
                    "--name", START_FROM, "--imgsz", str(IMGSZ), "--esp-detection", ESP_DETECTION, "--test-only",
                    "--result", "result_compare.json"], capture_output=True, text=True)
    before = json.load(open(f"{DET}/runs/{START_FROM}/result_compare.json"))
for k in ("synthetic", "real"):
    if k in res:
        e = res[k]
        print(f"{k:10s} {RUN:8s}: mAP50 {e['mAP50']:.3f} · recall {e['recall']:.3f} · precision {e['precision']:.3f}")
        if before and k in before:
            a = before[k]
            print(f"{'':10s} {START_FROM:8s}: mAP50 {a['mAP50']:.3f} · recall {a['recall']:.3f} · precision {a['precision']:.3f}  (before)")
if "real" not in res:
    print("⚠ no real test images - result measured on synthetic images only")

## 9 · Prepare for the ESP32 (export)
Saves the detector as `export/dice_det.onnx` with calibration and test images (`det_calib.npz`, `det_test.npz`,
`det_info.json`). The actual conversion into the ESP32 format happens afterwards **on the laptop**.

In [ ]:
import re, subprocess, time
t0 = time.time()
proc = subprocess.Popen(["python", f"{SCRIPTS}/det_export.py", "--weights", f"{DET}/runs/{RUN}/weights/best.pt", "--project", P, "--renders", RENDERS, "--esp-detection", ESP_DETECTION, "--imgsz", str(IMGSZ), "--output", EXPORT], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    z = re.sub(r"\x1b\[[0-9;]*[A-Za-z]", "", line.rstrip("\n").split("\r")[-1])   # no colour codes
    if z.strip() and (True):
        print(z, flush=True)
proc.wait()
print(f"⏱ {(time.time() - t0) / 60:.1f} min · return code {proc.returncode}")
for f in sorted(os.listdir(EXPORT)):
    print(f"  {f:24s} {os.path.getsize(os.path.join(EXPORT, f)) / 1024:8.1f} KB")

## What next?
Open **`notebooks/4_classifier.ipynb`**. When both are done, download the folder `export` from Drive to the laptop -
how is described in the trainer on the page **“Training & device”**.

## End the session (optional)
A connected Colab session uses **compute units even when it does nothing**. If you leave while something is still
running: set `END_SESSION = True` below and start this cell with ▶ - it queues up and ends the session when all cells
before it are done. All results are on Drive, nothing is lost.
**Note:** if a cell before it stops with an error, Colab discards the waiting cells - this one too. The session then
keeps running until Colab disconnects it (or you choose *Runtime → Disconnect and delete runtime*).
With *Run all* nothing happens here as long as `END_SESSION = False`.

In [ ]:
END_SESSION = False   # <- set to True and press ▶ if the session should end after the running cells
if END_SESSION:
    from google.colab import runtime
    print("Ending the session - results are on Drive.")
    runtime.unassign()
else:
    print("Session keeps running (END_SESSION = False).")